## Cell: `load_data`
Load all 3 core CSVs. Outputs: `epa_df`, `monitors_df`, `mismatches_df`.

> **Zerve:** Name this cell `load_data` in the canvas label field.

In [ ]:
# cell: load_data
import pandas as pd
import numpy as np

epa_df = pd.read_csv('data/epa_daily_aqi.csv')
epa_df = epa_df.dropna(subset=['aqi'])  # drop ~100k rows with null AQI
epa_df['date'] = pd.to_datetime(epa_df['date'])

monitors_df = pd.read_csv('data/epa_monitors.csv')

mismatches_df = pd.read_csv('data/station_mismatches.csv')
mismatches_df['date'] = pd.to_datetime(mismatches_df['date'])

print(f'EPA daily records (valid AQI): {len(epa_df):,}')
print(f'EPA monitoring stations: {len(monitors_df):,}')
print(f'Confirmed blind spot pairs: {len(mismatches_df):,}')
print(f'Date range: {epa_df["date"].min().date()} -> {epa_df["date"].max().date()}')

## Cell: `data_summary`
Print record counts, date range, station count, AQI distribution, and top 5 worst days.

> **Zerve:** Name this cell `data_summary`.

In [ ]:
# cell: data_summary
aqi_cats = pd.cut(
    epa_df['aqi'],
    bins=[0, 50, 100, 150, 200, 300, 501],
    labels=['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous'],
    right=True
)
summary = aqi_cats.value_counts().sort_index()

print('=== GHOST AIR - Data Summary ===')
print(f'\nStations: {epa_df["site_name"].nunique():,}')
print(f'States covered: {epa_df["state"].nunique()}')
print(f'Date range: {epa_df["date"].min().date()} -> {epa_df["date"].max().date()}')
print(f'\nAQI Distribution:')
for cat, count in summary.items():
    pct = count / len(epa_df) * 100
    print(f'  {str(cat):35s}: {count:6,} ({pct:.1f}%)')

print(f'\nTop 5 worst days (stations reporting AQI >= 150):')
worst = epa_df[epa_df['aqi'] >= 150].groupby('date').size().sort_values(ascending=False).head(5)
for date, count in worst.items():
    print(f'  {str(date)[:10]}: {count} stations reporting Unhealthy+')

## Cell: `aqi_distribution`
Plot AQI distribution histogram with EPA category color bands. Output: `aqi_chart`.

> **Zerve:** Name this cell `aqi_distribution`.

In [ ]:
# cell: aqi_distribution
import plotly.express as px
import plotly.graph_objects as go

aqi_chart = px.histogram(
    epa_df[epa_df['aqi'] <= 300],
    x='aqi',
    nbins=60,
    color_discrete_sequence=['#3b82f6'],
    title='AQI Distribution - 991 EPA Stations, Summer 2023',
    labels={'aqi': 'Air Quality Index (AQI)', 'count': 'Station-Days'},
    template='plotly_white'
)

bands = [
    (0,   50,  '#22c55e', 'Good'),
    (50,  100, '#eab308', 'Moderate'),
    (100, 150, '#f97316', 'USG'),
    (150, 200, '#ef4444', 'Unhealthy'),
    (200, 300, '#a855f7', 'Very Unhealthy'),
]
for lo, hi, color, label in bands:
    aqi_chart.add_vrect(
        x0=lo, x1=hi, fillcolor=color, opacity=0.1,
        annotation_text=label, annotation_position='top left'
    )

aqi_chart.update_layout(height=420, xaxis_range=[0, 300])
aqi_chart.show()

## Cell: `station_map`
Plot all 991 EPA stations as green dots on a US map using pydeck. Output: `station_map_fig`.

> **Zerve:** Name this cell `station_map`.

In [ ]:
# cell: station_map
import pydeck as pdk

station_map_fig = pdk.Deck(
    layers=[
        pdk.Layer(
            'ScatterplotLayer',
            data=monitors_df,
            get_position=['lon', 'lat'],
            get_fill_color=[34, 197, 94, 180],
            get_radius=30000,
            pickable=True,
        )
    ],
    initial_view_state=pdk.ViewState(latitude=39.5, longitude=-98.35, zoom=3.5),
    tooltip={'text': '{site_name}\n{state}'},
    map_style='mapbox://styles/mapbox/light-v9'
)

print(f'Mapping {len(monitors_df):,} EPA PM2.5 monitoring stations')
station_map_fig

## Cell: `blind_spot_analysis`
Summarize the 24 confirmed blind spot pairs: stats and ranked table. Output: `blind_spots_df`.

> **Zerve:** Name this cell `blind_spot_analysis`.

In [ ]:
# cell: blind_spot_analysis
blind_spots_df = mismatches_df.copy()

print(f'Confirmed blind spot pairs: {len(blind_spots_df)}')
print(f'Unique high-AQI states: {blind_spots_df["high_state"].nunique()}')
print(f'Average distance between pairs: {blind_spots_df["distance_miles"].mean():.1f} miles')
print(f'Average AQI gap: {blind_spots_df["aqi_gap"].mean():.1f} points')
print(f'Max AQI gap: {blind_spots_df["aqi_gap"].max():.0f} points')

print(f'\nTop 10 most dramatic blind spots (by AQI gap):')
print(f'{"Date":12s} {"High Station":35s} {"AQI":>5s}  {"Low Station":30s} {"AQI":>5s}  {"Miles":>6s}  {"Gap":>5s}')
print('-' * 105)
top = blind_spots_df.sort_values('aqi_gap', ascending=False).head(10)
for _, row in top.iterrows():
    print(f'{str(row["date"])[:10]:12s} {str(row["high_station"])[:34]:35s} {row["high_aqi"]:5.0f}  '
          f'{str(row["low_station"])[:29]:30s} {row["low_aqi"]:5.0f}  '
          f'{row["distance_miles"]:6.0f}  {row["aqi_gap"]:5.0f}')


## Cell: `albany_case_study`
Filter the Albany-Burlington mismatch from June 8, 2023. Output: `albany_data` dict.

> **Zerve:** Name this cell `albany_case_study`.

In [ ]:
# cell: albany_case_study
albany_rows = mismatches_df[
    mismatches_df['high_station'].str.contains('ALBANY', case=False, na=False)
].sort_values('aqi_gap', ascending=False)

if len(albany_rows) > 0:
    row = albany_rows.iloc[0]
    albany_data = {
        'date': str(row['date'])[:10],
        'high_station': row['high_station'],
        'high_state': row['high_state'],
        'high_aqi': float(row['high_aqi']),
        'low_station': row['low_station'],
        'low_state': row['low_state'],
        'low_aqi': float(row['low_aqi']),
        'distance_miles': float(row['distance_miles']),
        'aqi_gap': float(row['aqi_gap']),
        'region': 'Adirondack Region, NY'
    }
else:
    # Hardcoded fallback -- always works for demo
    albany_data = {
        'date': '2023-06-08',
        'high_station': 'ALBANY COUNTY HEALTH DEPT',
        'high_state': 'New York',
        'high_aqi': 166.0,
        'low_station': 'ZAMPIERI STATE OFFICE BUILDING',
        'low_state': 'Vermont',
        'low_aqi': 11.0,
        'distance_miles': 129.8,
        'aqi_gap': 155.0,
        'region': 'Adirondack Region, NY'
    }

print('=' * 62)
print('ALBANY CASE STUDY')
print('=' * 62)
print(f'  Date          : {albany_data["date"]}')
print(f'  High station  : {albany_data["high_station"]} ({albany_data["high_state"]})')
print(f'  AQI           : {albany_data["high_aqi"]:.0f}  -- UNHEALTHY')
print(f'  Low station   : {albany_data["low_station"]} ({albany_data["low_state"]})')
print(f'  AQI           : {albany_data["low_aqi"]:.0f}   -- GOOD')
print(f'  Distance      : {albany_data["distance_miles"]:.0f} miles apart')
print(f'  AQI gap       : {albany_data["aqi_gap"]:.0f} points')
print(f'  Unmonitored   : {albany_data["region"]}')
print('=' * 62)

## Cell: `albany_comparison_chart`
Side-by-side bar chart: Albany AQI 166 (red) vs Burlington AQI 11 (green). Output: `albany_chart`.

> **Zerve:** Name this cell `albany_comparison_chart`.

In [ ]:
# cell: albany_comparison_chart
import plotly.graph_objects as go

high_label = albany_data['high_station'].title()
low_label = albany_data['low_station'].title()

albany_chart = go.Figure()

albany_chart.add_trace(go.Bar(
    x=[high_label],
    y=[albany_data['high_aqi']],
    name='Unhealthy',
    marker_color='#dc2626',
    text=[f'AQI {albany_data["high_aqi"]:.0f}<br>UNHEALTHY'],
    textposition='outside',
    width=0.35
))

albany_chart.add_trace(go.Bar(
    x=[low_label],
    y=[albany_data['low_aqi']],
    name='Good',
    marker_color='#16a34a',
    text=[f'AQI {albany_data["low_aqi"]:.0f}<br>GOOD'],
    textposition='outside',
    width=0.35
))

albany_chart.update_layout(
    title=(
        f'Albany Case Study -- {albany_data["date"]} | '
        f'{albany_data["distance_miles"]:.0f} miles apart | '
        f'AQI gap: {albany_data["aqi_gap"]:.0f} points'
    ),
    yaxis_title='Air Quality Index (AQI)',
    yaxis_range=[0, 210],
    template='plotly_white',
    height=460,
    showlegend=False,
    bargap=0.5,
    annotations=[
        dict(
            x=0.5, y=0.96, xref='paper', yref='paper',
            text=f'The entire {albany_data["region"]} between these stations had ZERO air quality monitoring',
            showarrow=False,
            font=dict(size=12, color='#374151'),
            bgcolor='#fef3c7', bordercolor='#d97706', borderwidth=1
        )
    ]
)
albany_chart.show()

## Cell: `blind_spot_map`
US map: green dots (all stations), large red dots (blind spot stations). Output: `blind_spot_map_fig`.

> **Zerve:** Name this cell `blind_spot_map`.

In [ ]:
# cell: blind_spot_map
import pydeck as pdk
import pandas as pd

blind_highs = mismatches_df[['high_lat', 'high_lon', 'high_station', 'high_aqi']].rename(
    columns={'high_lat': 'lat', 'high_lon': 'lon', 'high_station': 'site_name', 'high_aqi': 'aqi'}
)
blind_lows = mismatches_df[['low_lat', 'low_lon', 'low_station', 'low_aqi']].rename(
    columns={'low_lat': 'lat', 'low_lon': 'lon', 'low_station': 'site_name', 'low_aqi': 'aqi'}
)
blind_stations = pd.concat([blind_highs, blind_lows]).drop_duplicates('site_name')
blind_names = set(blind_stations['site_name'])

monitors_plot = monitors_df.copy()
monitors_plot['color'] = monitors_plot['site_name'].apply(
    lambda x: [220, 38, 38, 210] if x in blind_names else [34, 197, 94, 140]
)
monitors_plot['radius'] = monitors_plot['site_name'].apply(
    lambda x: 65000 if x in blind_names else 22000
)

blind_spot_map_fig = pdk.Deck(
    layers=[
        pdk.Layer(
            'ScatterplotLayer',
            data=monitors_plot,
            get_position=['lon', 'lat'],
            get_fill_color='color',
            get_radius='radius',
            pickable=True,
        )
    ],
    initial_view_state=pdk.ViewState(latitude=41.5, longitude=-76.0, zoom=5),
    tooltip={'text': '{site_name}'},
    map_style='mapbox://styles/mapbox/light-v9'
)

print(f'Red stations (involved in blind spot pairs): {len(blind_stations)}')
print(f'Green stations (adequate coverage): {len(monitors_df) - len(blind_names)}')
blind_spot_map_fig

## Cell: `coverage_gaps`
Load `epa_coverage_gaps.csv` and print the top 10 most isolated EPA stations. Output: `isolation_table`.

> **Zerve:** Name this cell `coverage_gaps`.

In [ ]:
# cell: coverage_gaps
import pandas as pd

gaps_df = pd.read_csv('data/epa_coverage_gaps.csv')

isolation_table = (
    gaps_df
    .sort_values('nearest_station_miles', ascending=False)
    .head(10)[['site_name', 'state', 'county', 'nearest_station_miles']]
    .reset_index(drop=True)
)
isolation_table.columns = ['Station', 'State', 'County', 'Miles to Nearest Station']
isolation_table['Miles to Nearest Station'] = isolation_table['Miles to Nearest Station'].round(1)

print('Top 10 Most Isolated EPA Stations')
print('(furthest distance to nearest neighboring station)')
print(isolation_table.to_string(index=False))